# GIAI ĐOẠN 0 – TV2
## Tiền xử lý dữ liệu
**Đề tài:** Phân tích giỏ hàng siêu thị và xây dựng gợi ý mua kèm  
**Dataset gốc:** `Groceries_dataset.csv`

**Nhiệm vụ của TV2** (Bước 4 trong phân công): xử lý các vấn đề mà TV1 đã ghi nhận ở phần đánh giá chất lượng dữ liệu, cụ thể:
1. Quyết định cách xử lý các dòng trùng
2. Chuyển cột `Date` sang kiểu `datetime`.
3. Chuẩn hóa tên mặt hàng (`itemDescription`).
4. Kiểm tra và xử lý ngoại lệ (nếu có).
5. Xuất ra file dữ liệu đã làm sạch để bàn giao cho TV3.


## 0. Đọc dữ liệu gốc

In [3]:
import pandas as pd
import numpy as np

# Nếu chạy trên Colab và chưa có file, mở comment 2 dòng dưới để upload
# from google.colab import files

df = pd.read_csv("/content/.config/logs/Groceries_dataset.csv.zip")
print("Kích thước dữ liệu gốc:", df.shape)
df.head()

Kích thước dữ liệu gốc: (38765, 3)


,Member_number,Date,itemDescription
0,1808,21-07-2015,tropical fruit
1,2552,05-01-2015,whole milk
2,2300,19-09-2015,pip fruit
3,1187,12-12-2015,other vegetables
4,3037,01-02-2015,whole milk


## 1. Xử lý dòng trùng

TV1 ghi nhận: dataset không có cột `Quantity`, nên các dòng trùng hoàn toàn (cùng khách hàng, cùng ngày, cùng mặt hàng) nhiều khả năng là khách hàng mua **nhiều hơn 1 đơn vị** của cùng sản phẩm, chứ không hẳn là lỗi dữ liệu.

Cách xử lý: thay vì xóa các dòng trùng (sẽ làm mất thông tin số lượng thực tế), ta **gộp chúng lại** và tạo cột `Quantity` để lưu lại số lần mua. Đây là cách xử lý giữ được nhiều thông tin nhất, và cũng là bước cần thiết trước khi TV3 chuyển sang dạng transaction.

In [4]:
so_dong_truoc = df.shape[0]

# Đếm số lần mỗi (khách hàng, ngày, mặt hàng) xuất hiện -> Quantity
df_clean = df.copy()
df_clean["Quantity"] = df_clean.groupby(
    ["Member_number", "Date", "itemDescription"]
)["itemDescription"].transform("count")

# Gộp các dòng trùng lại thành 1 dòng đại diện (đã có Quantity)
df_clean = df_clean.drop_duplicates(
    subset=["Member_number", "Date", "itemDescription"]
).reset_index(drop=True)

so_dong_sau = df_clean.shape[0]

print("Số dòng trước khi xử lý trùng:", so_dong_truoc)
print("Số dòng sau khi gộp trùng:", so_dong_sau)
print("Số dòng đã gộp (Quantity > 1):", int((df_clean["Quantity"] > 1).sum()))
df_clean.head()

Số dòng trước khi xử lý trùng: 38765
Số dòng sau khi gộp trùng: 38006
Số dòng đã gộp (Quantity > 1): 732


,Member_number,Date,itemDescription,Quantity
0,1808,21-07-2015,tropical fruit,1
1,2552,05-01-2015,whole milk,1
2,2300,19-09-2015,pip fruit,1
3,1187,12-12-2015,other vegetables,1
4,3037,01-02-2015,whole milk,1


## 2. Chuyển `Date` sang datetime

TV1 đã xác nhận toàn bộ giá trị `Date` hợp lệ theo định dạng `dd-mm-yyyy`, nên ở đây chuyển thẳng sang `datetime` mà không cần `errors="coerce"`.

In [14]:
df_clean["Date"] = pd.to_datetime(df_clean["Date"], format="%d-%m-%Y")

print("Kiểu dữ liệu Date sau khi chuyển:", df_clean["Date"].dtype)
print("Khoảng thời gian dữ liệu:", df_clean["Date"].min().date(), "->", df_clean["Date"].max().date())

Kiểu dữ liệu Date sau khi chuyển: datetime64[ns]
Khoảng thời gian dữ liệu: 2014-01-01 -> 2015-12-30


## 3. Chuẩn hóa tên mặt hàng (`itemDescription`)

Chuẩn hóa cơ bản: bỏ khoảng trắng đầu/cuối, đưa về chữ thường, để đảm bảo cùng 1 mặt hàng không bị tách thành nhiều tên khác nhau khi TV3 xây dựng transaction.

In [15]:
so_ten_truoc = df_clean["itemDescription"].nunique()

df_clean["itemDescription"] = (
    df_clean["itemDescription"]
    .astype(str)
    .str.strip()
    .str.lower()
)

so_ten_sau = df_clean["itemDescription"].nunique()

print("Số tên mặt hàng trước chuẩn hóa:", so_ten_truoc)
print("Số tên mặt hàng sau chuẩn hóa:", so_ten_sau)

if so_ten_truoc != so_ten_sau:
    print("-> Đã gộp được các biến thể viết hoa/thường hoặc khoảng trắng thừa.")
else:
    print("-> Không phát hiện thêm biến thể trùng lặp sau chuẩn hóa cơ bản.")

Số tên mặt hàng trước chuẩn hóa: 167
Số tên mặt hàng sau chuẩn hóa: 167
-> Không phát hiện thêm biến thể trùng lặp sau chuẩn hóa cơ bản.


### Kiểm tra thêm: các tên mặt hàng gần giống nhau (không tự động gộp)

Bước này chỉ để **rà soát bằng mắt** xem có mặt hàng nào viết khác nhau nhưng cùng nghĩa không (ví dụ số ít/số nhiều, viết tắt...). Nếu phát hiện, nhóm cần thống nhất quy tắc gộp trước khi bàn giao cho TV3, tránh tự ý đổi tên mà cả nhóm chưa duyệt.

In [16]:
from difflib import SequenceMatcher

ten_mat_hang = sorted(df_clean["itemDescription"].unique())
canh_bao = []

for i in range(len(ten_mat_hang)):
    for j in range(i + 1, len(ten_mat_hang)):
        ty_le_giong = SequenceMatcher(None, ten_mat_hang[i], ten_mat_hang[j]).ratio()
        if ty_le_giong > 0.9:
            canh_bao.append((ten_mat_hang[i], ten_mat_hang[j], round(ty_le_giong, 2)))

print("Số cặp tên mặt hàng nghi ngờ giống nhau (>90%):", len(canh_bao))
for cap in canh_bao[:20]:
    print(cap)

Số cặp tên mặt hàng nghi ngờ giống nhau (>90%): 0


## 4. Kiểm tra và xử lý ngoại lệ `Member_number`

In [17]:
mask_khong_hop_le = df_clean["Member_number"].isna() | (df_clean["Member_number"] <= 0)
so_luong_khong_hop_le = int(mask_khong_hop_le.sum())

print("Số Member_number không hợp lệ:", so_luong_khong_hop_le)

if so_luong_khong_hop_le > 0:
    df_clean = df_clean[~mask_khong_hop_le].reset_index(drop=True)
    print("-> Đã loại bỏ các dòng có Member_number không hợp lệ.")
else:
    print("-> Không có Member_number nào cần loại bỏ.")

Số Member_number không hợp lệ: 0
-> Không có Member_number nào cần loại bỏ.


## 5. So sánh trước / sau xử lý

In [18]:
bang_so_sanh = pd.DataFrame({
    "Tieu_chi": [
        "Số dòng",
        "Số khách hàng khác nhau",
        "Số mặt hàng khác nhau",
        "Kiểu dữ liệu Date",
        "Có cột Quantity"
    ],
    "Truoc_xu_ly": [
        df.shape[0],
        df["Member_number"].nunique(),
        df["itemDescription"].nunique(),
        str(df["Date"].dtype),
        "Không"
    ],
    "Sau_xu_ly": [
        df_clean.shape[0],
        df_clean["Member_number"].nunique(),
        df_clean["itemDescription"].nunique(),
        str(df_clean["Date"].dtype),
        "Có"
    ]
})

bang_so_sanh

,Tieu_chi,Truoc_xu_ly,Sau_xu_ly
0,Số dòng,38765,38006
1,Số khách hàng khác nhau,3898,3898
2,Số mặt hàng khác nhau,167,167
3,Kiểu dữ liệu Date,object,datetime64[ns]
4,Có cột Quantity,Không,Có


## 6. Xuất file đã làm sạch để bàn giao cho TV3

In [19]:
df_clean.to_csv("Groceries_cleaned.csv", index=False)
print("Đã lưu file: Groceries_cleaned.csv")
print("Kích thước dữ liệu sau xử lý:", df_clean.shape)
df_clean.head()

Đã lưu file: Groceries_cleaned.csv
Kích thước dữ liệu sau xử lý: (38006, 4)


,Member_number,Date,itemDescription,Quantity
0,1808,2015-07-21,tropical fruit,1
1,2552,2015-01-05,whole milk,1
2,2300,2015-09-19,pip fruit,1
3,1187,2015-12-12,other vegetables,1
4,3037,2015-02-01,whole milk,1


## 7. Ghi chú bàn giao cho TV3

File `Groceries_cleaned.csv` khác với file gốc ở các điểm sau:

- Đã gộp các dòng trùng (cùng khách hàng/ngày/mặt hàng) thành 1 dòng, có thêm cột **`Quantity`** thể hiện số lượng mua.
- Cột **`Date`** đã ở dạng `datetime` khi đọc lại trong Python (khi lưu ra `.csv` sẽ trở về dạng chuỗi `YYYY-MM-DD`, cần `pd.to_datetime` lại khi đọc file này).
- Cột **`itemDescription`** đã được `strip()` và chuyển về chữ thường.
- Đã loại các `Member_number` không hợp lệ (nếu có).